In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time


# Nonlinear Dynamics & Simulation
def cartpole_dynamics(state, u, params):
    x, x_dot, theta, theta_dot = state
    mc, mp, l, g = params
    
    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)
    
    theta_ddot = (
        g * sin_theta + cos_theta * (-mp * l * theta_dot**2 * sin_theta - u) / (mc + mp)
    ) / (l * (4/3 - mp * cos_theta**2 / (mc + mp))) 

    x_ddot = (
        mp * l * (theta_dot**2 * sin_theta - theta_ddot * cos_theta) + u
    ) / (mc + mp)
    
    return np.array([x_dot, x_ddot, theta_dot, theta_ddot])

def f(x, u, params, dt):
    # Discrete dynamics: x_next = x + dt*f(x,u)
    return x + dt * cartpole_dynamics(x, u, params)

def simulate_with_control(initial_state, u_sequence, params, dt):
    state = np.array(initial_state)
    trajectory = [state]
    for u in u_sequence:
        state = f(state, u, params, dt)
        trajectory.append(state)
    return np.array(trajectory)


# iLQR Helper Functions
def compute_total_cost(x_traj, u_seq, Q, R):
    # x_traj: (horizon+1, n), u_seq: (horizon,)
    cost = 0
    horizon = len(u_seq)
    for t in range(horizon):
        x = x_traj[t]
        u = u_seq[t]
        cost += x.T @ Q @ x + u * R * u
    cost += x_traj[-1].T @ Q @ x_traj[-1]
    return cost

def linearize_dynamics(x, u, params, dt, eps=1e-5):
    # Finite differences to compute A = df/dx and B = df/du for f(x,u) = x + dt*cartpole_dynamics(x,u)
    n = len(x)
    A = np.zeros((n, n))
    B = np.zeros((n, 1))
    # Compute A
    for i in range(n):
        dx = np.zeros(n)
        dx[i] = eps
        f_plus = f(x + dx, u, params, dt)
        f_minus = f(x - dx, u, params, dt)
        A[:, i] = (f_plus - f_minus) / (2 * eps)
    # Compute B (control is scalar)
    du = eps
    f_plus = f(x, u + du, params, dt)
    f_minus = f(x, u - du, params, dt)
    B[:, 0] = (f_plus - f_minus) / (2 * du)
    return A, B

def ilqr(initial_state, u_init, Q, R, horizon, params, dt, max_iter=100, tol=1e-6):
    # Dimensions
    n = len(initial_state)  # state dim (4)
    m = 1                  # control dim (scalar)
    
    # Forward pass: compute nominal trajectory using current control sequence
    x_traj = [initial_state.copy()]
    for t in range(horizon):
        x_next = f(x_traj[t], u_init[t], params, dt)
        x_traj.append(x_next)
    x_traj = np.array(x_traj)
    cost_prev = compute_total_cost(x_traj, u_init, Q, R)
    
    for iteration in range(max_iter):
        # Storage for linearized dynamics
        A_list = []
        B_list = []
        for t in range(horizon):
            A, B = linearize_dynamics(x_traj[t], u_init[t], params, dt)
            A_list.append(A)
            B_list.append(B)
        
        # Backward pass
        # Terminal cost derivatives
        x_final = x_traj[-1]
        V_x = 2 * Q @ x_final       # shape (n,)
        V_xx = 2 * Q                # shape (n,n)
        
        k_list = [None] * horizon  # feedforward terms (each shape (m,))
        K_list = [None] * horizon  # feedback gains (each shape (m,n))
        diverged = False
        
        # Backward recursion (from horizon-1 to 0)
        for t in reversed(range(horizon)):
            x_t = x_traj[t]
            u_t = u_init[t]
            # Stage cost derivatives: L(x,u) = x^T Q x + u R u
            L_x = 2 * Q @ x_t
            L_u = 2 * R * u_t
            L_xx = 2 * Q
            L_uu = 2 * R
            L_ux = np.zeros((m, n))
            
            A = A_list[t]
            B = B_list[t]
            
            Q_x = L_x + A.T @ V_x
            Q_u = L_u + B.T @ V_x  # shape (1,)
            Q_xx = L_xx + A.T @ V_xx @ A
            Q_ux = L_ux + B.T @ V_xx @ A
            Q_uu = L_uu + B.T @ V_xx @ B  # shape (1,1)
            
            # Regularize Q_uu to ensure positive definiteness
            Q_uu_reg = Q_uu + 1e-6 * np.eye(m)
            try:
                inv_Q_uu = np.linalg.inv(Q_uu_reg)
            except np.linalg.LinAlgError:
                diverged = True
                break
            
            k = - inv_Q_uu @ Q_u  # feedforward term (shape (1,))
            K = - inv_Q_uu @ Q_ux  # feedback gain (shape (1,n))
            
            k_list[t] = k
            K_list[t] = K
            
            V_x = Q_x + K.T @ Q_uu @ k + K.T @ Q_u + Q_ux.T @ k
            V_xx = Q_xx + K.T @ Q_uu @ K + K.T @ Q_ux + Q_ux.T @ K
            V_xx = 0.5 * (V_xx + V_xx.T)  # ensure symmetry
        
        if diverged:
            break
        
        # Forward pass with line search to update control sequence
        alpha = 1.0
        found_better = False
        for ls in range(10):
            x_new = [initial_state.copy()]
            u_new = []
            for t in range(horizon):
                delta_x = x_new[t] - x_traj[t]
                du = alpha * k_list[t] + K_list[t] @ delta_x
                u_new_t = u_init[t] + du[0]  # scalar update
                u_new_t = np.clip(u_new_t, u_bounds[0], u_bounds[1])
                u_new.append(u_new_t)
                x_next = f(x_new[t], u_new_t, params, dt)
                x_new.append(x_next)
            u_new = np.array(u_new)
            x_new = np.array(x_new)
            cost_new = compute_total_cost(x_new, u_new, Q, R)
            if cost_new < cost_prev:
                found_better = True
                break
            alpha *= 0.5
        
        if not found_better:
            break
        
        if abs(cost_prev - cost_new) < tol:
            u_init = u_new
            x_traj = x_new
            cost_prev = cost_new
            break
        
        u_init = u_new
        x_traj = x_new
        cost_prev = cost_new
    return u_init, x_traj

# Receding Horizon iLQR Simulation
def simulate_ilqr(initial_state, Q, R, horizon, sim_time, params, dt, u_bounds=(-10, 10), max_iter=100, tol=1e-6):
    state = np.array(initial_state)
    trajectory = [state]
    controls = []
    steps = int(sim_time / dt)
    for _ in range(steps):
        # Initialize control sequence (zeros)
        u_init = np.zeros(horizon)
        u_opt, x_traj = ilqr(state, u_init, Q, R, horizon, params, dt, max_iter, tol)
        u = u_opt[0]  # Apply only the first control action
        controls.append(u)
        state_dot = cartpole_dynamics(state, u, params)
        state = state + dt * state_dot
        trajectory.append(state)
    return np.array(trajectory), np.array(controls)

# Main Evaluation Loop
trajectories = np.load('fractional_system_trajectories.npy')
U_optimal = np.load('optimal_control_U.npy')
Q_matrices = np.load('LQR_Q.npy')
R_matrices = np.load('LQR_R.npy')

# System parameters
mc, mp, l, g = 1.0, 0.1, 1.0, 9.81
params = (mc, mp, l, g)
dt = 0.1
time_horizon = 16  # prediction horizon for iLQR
sim_time = 3.2
u_bounds = (-0.5, 0.5)

mses = []
maes = []

start_time = time.time()

for i in range(8000, 8050):
    x0 = trajectories[i, 0]
    Q = Q_matrices[i]
    R = R_matrices[i]
    
    trajectory_est, estimated_controls_est = simulate_ilqr(x0, Q, R, time_horizon, sim_time, params, dt, u_bounds)
    
    mae = mean_absolute_error(estimated_controls_est, U_optimal[i])
    mse = mean_squared_error(estimated_controls_est, U_optimal[i])
    print("Iteration", i, "MAE:", mae, "MSE:", mse)
    maes.append(mae)
    mses.append(mse)

end_time = time.time()
total_runtime = end_time - start_time
print("Total runtime: {:.2f} seconds".format(total_runtime))

Iteration 8000 MAE: 0.16171377508455906 MSE: 0.07238195187256277
Iteration 8001 MAE: 0.4934763558631703 MSE: 0.24763476313832505
Iteration 8002 MAE: 0.47752790724160654 MSE: 0.2412241422745785
Iteration 8003 MAE: 0.49625218272126315 MSE: 0.25034956453741786
Iteration 8004 MAE: 0.4689257479765802 MSE: 0.23346716362407516
Iteration 8005 MAE: 0.24316405260505286 MSE: 0.06725093538571666
Iteration 8006 MAE: 0.488662811464464 MSE: 0.2507515405227042
Iteration 8007 MAE: 0.5007591936776075 MSE: 0.25712513173012824
Iteration 8008 MAE: 0.4806931971714933 MSE: 0.24235782040817025
Iteration 8009 MAE: 0.4918971206733529 MSE: 0.2550589376600435
Iteration 8010 MAE: 0.5020873742102663 MSE: 0.2542740186842655
Iteration 8011 MAE: 0.5076180067961706 MSE: 0.26034245078597545
Iteration 8012 MAE: 0.4887182120472607 MSE: 0.2446294304582045
Iteration 8013 MAE: 0.5218905163577643 MSE: 0.2778907905013091
Iteration 8014 MAE: 0.4920929856049017 MSE: 0.247535238656066
Iteration 8015 MAE: 0.4008676633759756 MSE: 0